<a href="https://colab.research.google.com/github/Sahil-Nitjit/Alzheihmer-Detection-and-Progress-Prediction/blob/main/Copy_of_Untitled.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import zipfile

# Double-check the spelling of "Alzheimer" here to match your exact file name
zip_path = "/content/AugmentedAlzheimerDataset.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/mri_dataset")

print("Dataset Extracted")

Dataset Extracted


In [ ]:
import os

root = "/content/mri_dataset/AugmentedAlzheimerDataset"

for folder in os.listdir(root):
    print(folder)

NonDemented
MildDemented
ModerateDemented
VeryMildDemented


In [ ]:
import pandas as pd

history_df = pd.read_csv(
    "/content/alzheimer_patient_history_dataset_5000.csv"
)

print(history_df.shape)

history_df.head()

(5000, 21)


,patient_id,age,gender,education_level,family_history_alzheimers,genetic_risk,diabetes,hypertension,heart_disease,smoking_status,...,physical_activity,sleep_quality,depression_history,memory_loss_frequency,confusion_incidents,speech_difficulty,difficulty_walking,cognitive_test_score,mri_prediction,final_diagnosis
0,P00001,90,Female,Secondary,Yes,Yes,Yes,Yes,No,Never,...,Low,4,Yes,Rare,16,Yes,Yes,8.4,ModerateDemented,ModerateDemented
1,P00002,84,Female,Primary,Yes,No,No,No,Yes,Never,...,Low,2,No,Sometimes,3,No,No,11.1,ModerateDemented,ModerateDemented
2,P00003,88,Female,Primary,No,Yes,No,No,Yes,Current,...,Low,4,No,Sometimes,2,Yes,Yes,19.3,MildDemented,MildDemented
3,P00004,74,Female,Secondary,No,No,Yes,No,Yes,Current,...,Low,9,Yes,Sometimes,5,No,No,22.1,VeryMildDemented,VeryMildDemented
4,P00005,67,Male,Graduate,Yes,Yes,Yes,No,No,Former,...,Low,10,No,Frequent,6,No,No,17.6,MildDemented,MildDemented


In [ ]:
print(
    history_df["final_diagnosis"].value_counts()
)

final_diagnosis
MildDemented        1594
ModerateDemented    1467
VeryMildDemented    1009
NonDemented          930
Name: count, dtype: int64


In [ ]:
import os
import pandas as pd

fusion_rows = []

MRI_ROOT = "/content/mri_dataset/AugmentedAlzheimerDataset"

for diagnosis in [

    "NonDemented",
    "VeryMildDemented",
    "MildDemented",
    "ModerateDemented"

]:

    folder_path = os.path.join(
        MRI_ROOT,
        diagnosis
    )

    patient_subset = history_df[
        history_df["final_diagnosis"] == diagnosis
    ]

    if len(patient_subset) == 0:
        continue

    images = os.listdir(folder_path)

    print(
        diagnosis,
        "Images:",
        len(images)
    )

    for img in images:

        patient = patient_subset.sample(
            1
        ).iloc[0]

        row = patient.to_dict()

        row["image_path"] = os.path.join(
            folder_path,
            img
        )

        fusion_rows.append(row)

fusion_df = pd.DataFrame(
    fusion_rows
)

print(
    "Fusion Shape:",
    fusion_df.shape
)

NonDemented Images: 9600
VeryMildDemented Images: 8960
MildDemented Images: 8960
ModerateDemented Images: 6464
Fusion Shape: (33984, 22)


In [ ]:
fusion_df.to_csv(
    "/content/fusion_dataset.csv",
    index=False
)

print(
    "Saved Successfully"
)

Saved Successfully


In [ ]:
fusion_df.head()

,patient_id,age,gender,education_level,family_history_alzheimers,genetic_risk,diabetes,hypertension,heart_disease,smoking_status,...,sleep_quality,depression_history,memory_loss_frequency,confusion_incidents,speech_difficulty,difficulty_walking,cognitive_test_score,mri_prediction,final_diagnosis,image_path
0,P01116,66,Male,Secondary,Yes,Yes,No,No,Yes,Current,...,4,No,Rare,3,Yes,Yes,27.6,NonDemented,NonDemented,/content/mri_dataset/AugmentedAlzheimerDataset...
1,P01768,55,Male,Postgraduate,No,No,No,No,Yes,Current,...,1,No,Rare,10,No,Yes,26.2,NonDemented,NonDemented,/content/mri_dataset/AugmentedAlzheimerDataset...
2,P03163,54,Male,Postgraduate,Yes,No,No,No,Yes,Never,...,4,Yes,Rare,20,Yes,Yes,27.1,NonDemented,NonDemented,/content/mri_dataset/AugmentedAlzheimerDataset...
3,P02001,50,Male,Postgraduate,No,No,Yes,No,No,Current,...,10,No,Sometimes,6,No,No,26.9,NonDemented,NonDemented,/content/mri_dataset/AugmentedAlzheimerDataset...
4,P00943,74,Male,Postgraduate,Yes,Yes,Yes,Yes,No,Never,...,7,Yes,Rare,19,No,Yes,30.0,NonDemented,NonDemented,/content/mri_dataset/AugmentedAlzheimerDataset...


In [ ]:
print(fusion_df.shape)

fusion_df["final_diagnosis"].value_counts()

(33984, 22)


,count
final_diagnosis,
NonDemented,9600
VeryMildDemented,8960
MildDemented,8960
ModerateDemented,6464


In [ ]:
!pip install -q tensorflow pandas numpy scikit-learn

In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model

In [ ]:
fusion_df = pd.read_csv("/content/fusion_dataset.csv")

fusion_df.head()

,patient_id,age,gender,education_level,family_history_alzheimers,genetic_risk,diabetes,hypertension,heart_disease,smoking_status,...,sleep_quality,depression_history,memory_loss_frequency,confusion_incidents,speech_difficulty,difficulty_walking,cognitive_test_score,mri_prediction,final_diagnosis,image_path
0,P03561,61,Male,Graduate,Yes,No,No,No,No,Never,...,4,No,Rare,4,Yes,Yes,27.6,NonDemented,NonDemented,/content/mri_dataset/AugmentedAlzheimerDataset...
1,P01826,75,Male,Primary,No,No,No,Yes,No,Never,...,4,No,Rare,11,Yes,Yes,28.7,NonDemented,NonDemented,/content/mri_dataset/AugmentedAlzheimerDataset...
2,P01649,50,Male,Secondary,No,No,No,Yes,Yes,Never,...,3,Yes,Rare,2,No,No,26.6,NonDemented,NonDemented,/content/mri_dataset/AugmentedAlzheimerDataset...
3,P04503,72,Female,Primary,Yes,Yes,No,No,No,Former,...,4,No,Rare,7,Yes,Yes,28.1,NonDemented,NonDemented,/content/mri_dataset/AugmentedAlzheimerDataset...
4,P01239,59,Male,Postgraduate,No,No,No,No,No,Current,...,10,No,Rare,7,No,Yes,26.8,NonDemented,NonDemented,/content/mri_dataset/AugmentedAlzheimerDataset...


In [ ]:
categorical_cols = [

    "gender",
    "education_level",
    "family_history_alzheimers",
    "genetic_risk",
    "diabetes",
    "hypertension",
    "heart_disease",
    "smoking_status",
    "alcohol_consumption",
    "physical_activity",
    "sleep_quality",
    "depression_history",
    "speech_difficulty",
    "difficulty_walking",
    "mri_prediction"

]

for col in categorical_cols:

    le = LabelEncoder()

    fusion_df[col] = le.fit_transform(
        fusion_df[col].astype(str)
    )

In [ ]:
target_encoder = LabelEncoder()

fusion_df["target"] = target_encoder.fit_transform(
    fusion_df["final_diagnosis"]
)

print(target_encoder.classes_)

['MildDemented' 'ModerateDemented' 'NonDemented' 'VeryMildDemented']


In [ ]:
fusion_df["future_risk"] = (
    (
        fusion_df["genetic_risk"] +
        fusion_df["memory_loss_frequency"] +
        fusion_df["confusion_incidents"]
    ) > 5
).astype(int)

TypeError: unsupported operand type(s) for +: 'int' and 'str'

In [ ]:
print(fusion_df[[
    "genetic_risk",
    "memory_loss_frequency",
    "confusion_incidents"
]].dtypes)

genetic_risk              int64
memory_loss_frequency    object
confusion_incidents       int64
dtype: object


In [ ]:
print(fusion_df["memory_loss_frequency"].unique()[:20])

['Rare' 'Sometimes' 'Frequent']


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

fusion_df["memory_loss_frequency"] = le.fit_transform(
    fusion_df["memory_loss_frequency"].astype(str)
)

print(fusion_df["memory_loss_frequency"].head())

0    1
1    1
2    1
3    1
4    1
Name: memory_loss_frequency, dtype: int64


In [ ]:
fusion_df["future_risk"] = (
    fusion_df["genetic_risk"]
    + fusion_df["memory_loss_frequency"]
    + fusion_df["confusion_incidents"]
)

fusion_df["future_risk"] = (
    fusion_df["future_risk"] > 5
).astype(int)

In [ ]:
print(
    fusion_df[[
        "genetic_risk",
        "memory_loss_frequency",
        "confusion_incidents",
        "future_risk"
    ]].head()
)

   genetic_risk  memory_loss_frequency  confusion_incidents  future_risk
0             0                      1                    4            0
1             0                      1                   11            1
2             0                      1                    2            0
3             1                      1                    7            1
4             0                      1                    7            1


In [ ]:
tabular_features = [

    "age",
    "gender",
    "education_level",
    "family_history_alzheimers",
    "genetic_risk",
    "diabetes",
    "hypertension",
    "heart_disease",
    "smoking_status",
    "alcohol_consumption",
    "physical_activity",
    "sleep_quality",
    "depression_history",
    "memory_loss_frequency",
    "confusion_incidents",
    "speech_difficulty",
    "difficulty_walking",
    "cognitive_test_score",
    "mri_prediction"

]

In [ ]:
scaler = StandardScaler()

fusion_df[tabular_features] = scaler.fit_transform(
    fusion_df[tabular_features]
)

In [ ]:
train_df, test_df = train_test_split(

    fusion_df,

    test_size=0.2,

    stratify=fusion_df["target"],

    random_state=42

)

print(train_df.shape)
print(test_df.shape)

(27187, 24)
(6797, 24)


In [ ]:
IMG_SIZE = 224

In [ ]:
def load_image(path):

    img = tf.io.read_file(path)

    img = tf.image.decode_jpeg(img, channels=3)

    img = tf.image.resize(
        img,
        (IMG_SIZE, IMG_SIZE)
    )

    img = img / 255.0

    return img

In [ ]:
def create_dataset(df, batch_size=32):

    image_paths = df["image_path"].values

    tabular_data = df[tabular_features].values.astype(np.float32)

    diagnosis = df["target"].values

    risk = df["future_risk"].values

    def generator():

        for img_path, clinical, y1, y2 in zip(

            image_paths,
            tabular_data,
            diagnosis,
            risk

        ):

            yield (

                (
                    load_image(img_path),
                    clinical
                ),

                (
                    y1,
                    y2
                )

            )

    ds = tf.data.Dataset.from_generator(

        generator,

        output_signature=(

            (
                tf.TensorSpec(
                    shape=(224,224,3),
                    dtype=tf.float32
                ),

                tf.TensorSpec(
                    shape=(19,),
                    dtype=tf.float32
                )
            ),

            (
                tf.TensorSpec(
                    shape=(),
                    dtype=tf.int32
                ),

                tf.TensorSpec(
                    shape=(),
                    dtype=tf.int32
                )
            )

        )

    )

    return ds.batch(batch_size).prefetch(
        tf.data.AUTOTUNE
    )

In [ ]:
train_ds = create_dataset(train_df)

test_ds = create_dataset(test_df)

In [ ]:
image_input = Input(
    shape=(224,224,3),
    name="mri_input"
)

base_model = EfficientNetB0(

    include_top=False,

    weights="imagenet",

    input_tensor=image_input

)

base_model.trainable = False

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
x = base_model.output

x = GlobalAveragePooling2D()(x)

x = Dense(
    256,
    activation="relu"
)(x)

x = Dropout(0.3)(x)

image_features = Dense(
    128,
    activation="relu"
)(x)

In [ ]:
clinical_input = Input(
    shape=(19,),
    name="clinical_input"
)

In [ ]:
y = Dense(
    128,
    activation="relu"
)(clinical_input)

y = BatchNormalization()(y)

y = Dropout(0.3)(y)

y = Dense(
    64,
    activation="relu"
)(y)

clinical_features = Dense(
    32,
    activation="relu"
)(y)

In [ ]:
fusion = Concatenate()(
    [
        image_features,
        clinical_features
    ]
)

In [ ]:
fusion = Dense(
    256,
    activation="relu"
)(fusion)

fusion = BatchNormalization()(fusion)

fusion = Dropout(0.4)(fusion)

fusion = Dense(
    128,
    activation="relu"
)(fusion)

In [ ]:
diagnosis_output = Dense(

    4,

    activation="softmax",

    name="diagnosis_output"

)(fusion)

In [ ]:
diagnosis_output = Dense(

    4,

    activation="softmax",

    name="diagnosis_output"

)(fusion)

In [ ]:
risk_output = Dense(

    1,

    activation="sigmoid",

    name="risk_output"

)(fusion)

In [ ]:
model = Model(

    inputs=[
        image_input,
        clinical_input
    ],

    outputs=[
        diagnosis_output,
        risk_output
    ]

)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ mri_input           │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ mri_input[0][0]   │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 224, 224,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,499,592 (17.16 MB)

 Trainable params: 449,253 (1.71 MB)

 Non-trainable params: 4,050,339 (15.45 MB)

In [ ]:
model.compile(

    optimizer="adam",

    loss={

        "diagnosis_output":
            "sparse_categorical_crossentropy",

        "risk_output":
            "binary_crossentropy"

    },

    metrics={

        "diagnosis_output":
            "accuracy",

        "risk_output":
            "accuracy"

    }

)

In [ ]:
history = model.fit(

    train_ds,

    validation_data=test_ds,

    epochs=15

)

Epoch 1/15
    850/Unknown 187s 175ms/step - diagnosis_output_accuracy: 0.7998 - diagnosis_output_loss: 0.4650 - loss: 0.7848 - risk_output_accuracy: 0.8607 - risk_output_loss: 0.3198

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


850/850 ━━━━━━━━━━━━━━━━━━━━ 238s 235ms/step - diagnosis_output_accuracy: 0.9202 - diagnosis_output_loss: 0.1944 - loss: 0.4116 - risk_output_accuracy: 0.9069 - risk_output_loss: 0.2171 - val_diagnosis_output_accuracy: 0.9994 - val_diagnosis_output_loss: 0.0025 - val_loss: 0.1093 - val_risk_output_accuracy: 0.9482 - val_risk_output_loss: 0.1067
Epoch 2/15
850/850 ━━━━━━━━━━━━━━━━━━━━ 159s 187ms/step - diagnosis_output_accuracy: 0.9916 - diagnosis_output_loss: 0.0251 - loss: 0.1366 - risk_output_accuracy: 0.9552 - risk_output_loss: 0.1116 - val_diagnosis_output_accuracy: 1.0000 - val_diagnosis_output_loss: 2.8287e-04 - val_loss: 0.0665 - val_risk_output_accuracy: 0.9688 - val_risk_output_loss: 0.0661
Epoch 3/15
850/850 ━━━━━━━━━━━━━━━━━━━━ 160s 188ms/step - diagnosis_output_accuracy: 0.9947 - diagnosis_output_loss: 0.0155 - loss: 0.0931 - risk_output_accuracy: 0.9698 - risk_output_loss: 0.0776 - val_diagnosis_output_accuracy: 1.0000 - val_diagnosis_output_loss: 2.0261e-05 - val_loss: 0.

In [ ]:
model.save(
    "/content/alzheimer_multimodal_fusion.keras"
)